# Parte 2. Clasificación multilingüe con transformers

Este notebook ejecuta los módulos `parte2` del repositorio. La métrica oficial es accuracy. Se comparan XLM-R y mDeBERTa, aumentación y ensambles, con análisis por idioma y por procedencia.

Los resultados se generan al ejecutar cada fase. Antes de la entrega, conservar las salidas de las corridas y completar las interpretaciones y aportes del equipo. El holdout se consulta después de congelar candidatos.

En Kaggle activar GPU e Internet y adjuntar los tres CSV de Parte 2. En Colab seleccionar un runtime GPU y cargar los CSV. Guardar `artifacts/parte2` y `submissions/parte2` en almacenamiento persistente al terminar cada sesión.

In [1]:
from pathlib import Path
import os
import subprocess
import sys

# En la nube se clona la rama de implementación si el notebook está aislado.
REPO_URL = "https://github.com/lauraS191/Proyecto-Machine-Learning.git"
BRANCH = "parte2-deep-learning"
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "parte2/experiments.json").exists()), None)
if ROOT is None:
    ROOT = Path.cwd() / "Proyecto-Machine-Learning"
    if not ROOT.exists():
        subprocess.run(["git", "clone", "--branch", BRANCH, REPO_URL, str(ROOT)], check=True)
    if not (ROOT / "parte2/experiments.json").exists():
        raise RuntimeError("La carpeta existente no contiene Parte 2. Seleccionar la rama parte2-deep-learning.")
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
INSTALL_DEPENDENCIES = Path("/kaggle").exists() or "COLAB_RELEASE_TAG" in os.environ
if INSTALL_DEPENDENCIES:
    subprocess.run([sys.executable, "-m", "pip", "install", "-r", "parte2/requirements.txt"], check=True)
print("Repositorio preparado")


Repositorio preparado


## Datos y controles de ejecución

Las rutas pueden suministrarse mediante `PARTE2_TRAIN`, `PARTE2_EVAL` y `PARTE2_SAMPLE`. La búsqueda automática considera los archivos adjuntos en Kaggle, los cargados en Colab y Downloads en local. El comando de preparación comprueba los tamaños oficiales.

Los entrenamientos base y la variante ortográfica se ejecutan automáticamente si CUDA está disponible. La traducción requiere activar su bandera y revisar después la muestra. Las fases de congelación y refit se activan cuando se hayan tomado las decisiones con desarrollo.

In [ ]:
import json
import pandas as pd
import torch
from parte2.common import read_json

DATA = ROOT / "data/parte2"
PREPARED = ROOT / "artifacts/parte2/prepared"
RUNS = ROOT / "artifacts/parte2/runs"
ARTIFACTS = PREPARED.parent
SUBMISSIONS = ROOT / "submissions/parte2"
RUN_BASELINES = torch.cuda.is_available()
RUN_ORTHOGRAPHIC = torch.cuda.is_available()
RUN_TRANSLATIONS = False
RUN_TRANSLATION_TRAINING = False
RUN_COMBINED = False
RUN_REPEATS = False
RUN_ENSEMBLE = False
RUN_HOLDOUT = False
RUN_REFITS = False
print({"cuda": torch.cuda.is_available(), "torch": torch.__version__})

def cli(*arguments):
    command = [sys.executable, "-u", "-m", "parte2", *map(str, arguments)]
    process = subprocess.Popen(command, cwd=ROOT, stdout=subprocess.PIPE,
                               stderr=subprocess.STDOUT, text=True, bufsize=1,
                               env={**os.environ, "HF_HUB_DISABLE_PROGRESS_BARS": "1"})
    for line in process.stdout:
        print(line, end="", flush=True)
    if process.wait():
        raise subprocess.CalledProcessError(process.returncode, command)

def raw_file(kind):
    explicit = os.environ.get("PARTE2_" + kind.upper())
    if explicit:
        return Path(explicit)
    filename = {"train": "train.csv", "eval": "eval.csv", "sample": "sample_submission.csv"}[kind]
    if (DATA / filename).exists():
        return DATA / filename
    download = Path.home() / "Downloads" / filename.replace(".csv", " (1).csv")
    if download.exists():
        return download
    prefix = "sample_submission" if kind == "sample" else kind
    candidates = []
    for folder in [Path("/kaggle/input"), Path("/content")]:
        if folder.exists():
            candidates.extend(p for p in folder.rglob(prefix + "*.csv") if ".git" not in p.parts and "Proyecto-Machine-Learning" not in p.parts)
    if len(candidates) != 1:
        raise ValueError(f"Definir PARTE2_{kind.upper()}. Candidatos encontrados: {candidates}")
    return candidates[0]

if not (PREPARED / "manifest.json").exists():
    cli("prepare", "--train", raw_file("train"), "--eval", raw_file("eval"), "--sample", raw_file("sample"))
manifest = read_json(PREPARED / "manifest.json")
display(pd.read_csv(PREPARED / "distribution.csv").groupby(["split", "origin"])["n"].sum().unstack())
print("Fingerprint", manifest["fingerprint"])
display(pd.read_csv(PREPARED / "language_review.csv")[["id", "language"]].head(12))


## Cuatro corridas iniciales

Se ajustan dos modelos multilingües con learning rates de 1e-5 y 2e-5, tres épocas virtuales, batch efectivo 32 y selección por accuracy de desarrollo. La longitud aumenta de 256 a 384 cuando se trunca más del 1 % de las fuentes de entrenamiento.

Las variantes y los originales comparten el mismo presupuesto de actualizaciones para la comparación. Revisar tiempos y reservar 30 % de la cuota para repeticiones y refits.

In [3]:
def train_once(name, experiment, variants=(), seed=None, source_run=None):
    destination = RUNS / name
    if (destination / "run.json").exists():
        state = read_json(destination / "run.json")
        if state["status"] == "complete":
            return
        raise RuntimeError(f"Corrida interrumpida en {destination}. Conservar registros y usar otro nombre.")
    arguments = ["train", "--name", name, "--experiment", experiment]
    if variants:
        arguments += ["--variants", *variants]
    if seed is not None:
        arguments += ["--seed", seed]
    if source_run is not None:
        arguments += ["--full", "--source-run", source_run, "--selection", ARTIFACTS / "selection.json"]
    cli(*arguments)

if RUN_BASELINES:
    for experiment in read_json(ROOT / "parte2/experiments.json")["experiments"]:
        train_once(experiment, experiment)

def leaderboard_local():
    records = []
    for path in sorted(RUNS.glob("*/dev_metrics.json")):
        report = read_json(path)
        run = read_json(path.parent / "run.json")
        if run["status"] != "complete":
            continue
        records.append({"run": path.parent.name, "accuracy": report["accuracy"],
                        "accuracy_nuevo": report["subgroups"].get("origin", {}).get("nuevo", {}).get("accuracy"),
                        "f1_macro": report["f1_macro"], "seconds": run.get("elapsed_seconds"),
                        "kind": run["kind"], "best_epoch": run.get("best_epoch")})
    if not records:
        return pd.DataFrame()
    return pd.DataFrame(records).sort_values(["accuracy", "accuracy_nuevo", "seconds"], ascending=[False, False, True], na_position="last")

table = leaderboard_local()
display(table)
BASE_WINNER = None
if not table.empty:
    base = table[table["run"].isin(read_json(ROOT / "parte2/experiments.json")["experiments"])]
    if not base.empty:
        BASE_WINNER = base.iloc[0]["run"]
print("Mejor configuración base", BASE_WINNER)


""


Mejor configuración base None


## Aumentación ortográfica

Se mantienen el orden de palabras y la letra ñ. Cada variante se vincula por ID y hash a una fuente de entrenamiento. Documentar aquí la diferencia medida frente a la misma configuración sin aumentación.

In [4]:
ORTH_TRAIN = ARTIFACTS / "orth_train.csv"
if not ORTH_TRAIN.exists():
    cli("augment", "--strategy", "orthographic", "--output", ORTH_TRAIN)
if RUN_ORTHOGRAPHIC and BASE_WINNER:
    train_once(BASE_WINNER + "_orth", BASE_WINNER, [ORTH_TRAIN])
display(leaderboard_local())


""


## Traducción y auditoría humana

M2M100 genera hasta 2.000 traducciones de entrenamiento hacia en/pt/fr. Revisar el archivo `mt_train_review.csv`, completar `decision` con `accept` o `reject` y explicar rechazos en `reason`. Se exige completar la muestra y mantener una tasa de rechazo máxima de 5 %. Las variantes no revisadas individualmente heredan su etiqueta fuente.

Activar `RUN_TRANSLATION_TRAINING` después de completar y aprobar esta auditoría. La generación se puede continuar con el mismo comando y `--resume` si se interrumpe después de un guardado.

In [5]:
MT_TRAIN = ARTIFACTS / "mt_train.csv"
MT_APPROVED = ARTIFACTS / "mt_train_approved.csv"
if RUN_TRANSLATIONS and not MT_TRAIN.exists():
    cli("augment", "--strategy", "translation", "--limit", 2000, "--output", MT_TRAIN)
if RUN_TRANSLATION_TRAINING:
    if not MT_APPROVED.exists():
        cli("approve-translations", "--variants", MT_TRAIN,
            "--review", ARTIFACTS / "mt_train_review.csv", "--output", MT_APPROVED)
    train_once(BASE_WINNER + "_mt", BASE_WINNER, [MT_APPROVED])
if RUN_COMBINED:
    train_once(BASE_WINNER + "_both", BASE_WINNER, [ORTH_TRAIN, MT_APPROVED])
display(leaderboard_local())


""


## Repetición y ensamble

Repetir las dos configuraciones más fuertes con semilla 123. Comparar tres pesos de promedio de probabilidades. El ensamble requiere mejora de accuracy y estabilidad por idioma/procedencia. Un intervalo bootstrap de la ganancia ayuda a interpretar la incertidumbre.

In [6]:
if RUN_REPEATS:
    strongest = leaderboard_local().query("kind == 'transformer'").head(2)
    for name in strongest["run"]:
        run = read_json(RUNS / name / "run.json")
        experiment = next(key for key, value in read_json(ROOT / "parte2/experiments.json")["experiments"].items()
                          if value["model"] == run["config"]["model"] and value["learning_rate"] == run["config"]["learning_rate"])
        variants = [(RUNS / name / item["path"]).resolve() for item in run["augmentations"]]
        train_once(name + "_seed123", experiment, variants, seed=123)
if RUN_ENSEMBLE:
    strongest = leaderboard_local().query("kind == 'transformer'").head(2)["run"].tolist()
    if len(strongest) != 2:
        raise ValueError("Se requieren dos candidatos")
    if not (RUNS / "ensemble").exists():
        cli("compare", "--first", RUNS / strongest[0], "--second", RUNS / strongest[1], "--output", RUNS / "ensemble")
    display(read_json(RUNS / "ensemble/comparison.json"))
display(leaderboard_local())


""


## Resultados por idioma y análisis de errores

Revisar al menos 100 errores cuando existan. Registrar categoría y explicación. Separar negación, ironía, comparación, concesión, sentimiento implícito y mezcla de idiomas. Describir los patrones encontrados y qué experimento los abordó.

In [7]:
table = leaderboard_local()
if not table.empty:
    winner = table.iloc[0]["run"]
    report = read_json(RUNS / winner / "dev_metrics.json")
    for group, values in report["subgroups"].items():
        print(group)
        display(pd.DataFrame(values).T)
    display(pd.DataFrame(report["confusion_matrix"], index=report["labels"], columns=report["labels"]))
    errors = ARTIFACTS / (winner + "_errors.csv")
    if not errors.exists():
        cli("errors", "--run", RUNS / winner, "--output", errors)
    display(pd.read_csv(errors).head(10))


## Congelar candidatos y consultar holdout

Completar `FINAL_CANDIDATES` con al menos cinco configuraciones diferenciadas elegidas en desarrollo e incluir los componentes de cada ensamble. Esto permite producir los cinco envíos mínimos y comprobar los candidatos antes de refitarlos. Activar `RUN_HOLDOUT` cuando termine la búsqueda. La CLI registra una única consulta de la selección congelada y cierra nuevos entrenamientos de búsqueda sobre estas particiones.

In [8]:
FINAL_CANDIDATES = []
SELECTION = ARTIFACTS / "selection.json"
if RUN_HOLDOUT:
    if not FINAL_CANDIDATES:
        raise ValueError("Completar FINAL_CANDIDATES antes de abrir holdout")
    if not SELECTION.exists():
        cli("freeze", "--candidates", *[RUNS / name for name in FINAL_CANDIDATES], "--output", SELECTION)
    if not (PREPARED / "holdout_summary.json").exists():
        cli("holdout", "--selection", SELECTION)
    display(pd.DataFrame(read_json(PREPARED / "holdout_summary.json"))[["run", "n", "accuracy", "f1_macro"]])


## Refits, envíos y reproducción

Los candidatos se refitan con todos los ejemplos originales, las mismas estrategias y el número de épocas elegido en desarrollo. La aumentación full se genera después de congelar candidatos. Las traducciones full tienen otra muestra de revisión.

El siguiente bloque refita los transformers congelados y después los ensambles. Si existen variantes de traducción, generar y aprobar previamente `mt_full_approved.csv` con los comandos documentados en `parte2/README.md`.

In [9]:
if RUN_REFITS:
    selection = read_json(SELECTION)
    candidate_names = [item["run_name"] for item in selection["candidates"]]
    for name in candidate_names:
        run = read_json(RUNS / name / "run.json")
        if run["kind"] != "transformer":
            continue
        variants = []
        for entry in run["augmentations"]:
            strategy = entry["strategy"]
            if strategy == "orthographic":
                path = ARTIFACTS / "orth_full.csv"
                if not path.exists():
                    cli("augment", "--strategy", "orthographic", "--scope", "full", "--selection", SELECTION, "--output", path)
            else:
                path = ARTIFACTS / "mt_full_approved.csv"
                if not path.exists():
                    raise ValueError("Generar y auditar las traducciones full antes del refit")
            variants.append(path)
        train_once(name + "_full", BASE_WINNER or "xlmr_2e5", variants, source_run=RUNS / name)
    for name in candidate_names:
        run = read_json(RUNS / name / "run.json")
        if run["kind"] == "ensemble" and not (RUNS / (name + "_full")).exists():
            cli("full-ensemble", "--source-run", RUNS / name, "--selection", SELECTION,
                "--members", *[RUNS / (item["run_name"] + "_full") for item in run["components"]], "--name", name + "_full")
    for name in candidate_names:
        bundle = RUNS / (name + "_full") / "bundle"
        output = SUBMISSIONS / (name + ".csv")
        if not output.exists():
            cli("predict", "--bundle", bundle, "--output", output)
        cli("verify", "--bundle", bundle, "--submission", output)


## Kaggle, selección pública y sustentación

Registrar al equipo y consultar las reglas de la competencia nueva. Completar línea base, hora de cierre, límite diario y selección final. Realizar al menos cinco envíos diferentes y registrar cada score observado con `python -m parte2 register`.

El registro actualiza `best_public.json`. Entregar el paquete exacto de ese envío y conservar el CSV verificado. Comparar también contra la línea base privada cuando se publique el resultado.

| Campo | Valor por completar con evidencia |
|---|---|
| Competencia y equipo | Pendiente |
| Línea base de Parte 2 | Pendiente |
| Hora de cierre y zona horaria | Pendiente |
| Límite diario y selección final | Pendiente |
| Mejor envío público y referencia | Pendiente |
| Accuracy privada y percentil | Disponible al publicarse |

## Aportes individuales e interpretación

| Integrante | Trabajo y decisión defendible | Commit o experimento |
|---|---|---|
| Completar | Completar | Completar |

Explicar qué aporta la transferencia multilingüe, cómo se evitó fuga entre variantes, qué cambió con la aumentación y qué idiomas siguen siendo difíciles. Relacionar cada conclusión con una tabla o corrida ejecutada.

Las fuentes, revisiones, licencias y asistencia de IA se documentan en `parte2/REFERENCIAS.md`. Actualizar ese documento con el uso efectivo y las personas que revisaron las traducciones.
